# 01B — Common canonical adapter

**Outcome:** convert either valid sector pack into the same Tier-0
episode-aware `SPEC-CORE`, with `SPLITS` and physically separate
`SPEC-EVAL`.

This notebook contains no ONT, splitter, well, valve or native metric logic.


## 1. Setup and sector switch

Run the relevant 01A notebook first. Change only `SECTOR` to switch sources.
`AS_OF_TS` is optional; when set, only observations available at or before
that timestamp may enter the canonical run.


In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_SCHEMAS,
    CORE_VERSION,
    PACK_ENTITY_SCHEMA,
    PACK_EPISODE_SCHEMA,
    audit_core,
    core_content_hashes,
    finalise_pack,
    materialise_canonical,
    read_json,
    runtime_probe,
    validate_pack,
    write_json,
)

# Change only this value when moving between sectors.
SECTOR = os.getenv("ADAPTER_SECTOR", "telecom")
BUILD_CANONICAL = os.getenv("BUILD_CANONICAL", "1") == "1"
AS_OF_TS = os.getenv("CANONICAL_AS_OF_TS") or None

PACK_RUN_IDS = {
    "telecom": "telecom_pack_v0_4",
    "petrobras_3w": "real_wells_expanded_v0_4",
}
if SECTOR not in PACK_RUN_IDS:
    raise ValueError(f"Choose one of {list(PACK_RUN_IDS)}")

PACK_RUN_ID = os.getenv("ADAPTER_PACK_RUN_ID", PACK_RUN_IDS[SECTOR])
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", f"{SECTOR}_core_v0_7_episode_run1")
PACK_ROOT = DRIVE_ROOT / "outputs" / "packs" / SECTOR / PACK_RUN_ID
RUN_ROOT = DRIVE_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID

display(pd.Series({
    "sector": SECTOR,
    "pack_root": str(PACK_ROOT),
    "canonical_root": str(RUN_ROOT),
    "build": BUILD_CANONICAL,
    "as_of_ts": AS_OF_TS or "all observations",
}, name="value").to_frame())


## 2. Inspect and materialise the frozen contract

`SPEC-CORE` contains only telemetry, authored metric semantics,
observation-derived entity and episode bounds, plus periodic collection
gaps. Labels
stay in the physically separate `SPEC-EVAL` directory.


In [ ]:
display(pd.DataFrame([
    {"table": name, "columns": ", ".join(columns)}
    for name, columns in CORE_SCHEMAS.items()
]))

pack_manifest = validate_pack(PACK_ROOT)
display(pd.Series(pack_manifest, name="value").to_frame())

if BUILD_CANONICAL:
    workflow = materialise_canonical(
        PACK_ROOT,
        RUN_ROOT,
        include_evaluation=True,
        as_of_ts=AS_OF_TS,
    )
else:
    workflow = read_json(RUN_ROOT / "workflow_report.json")

core_audit = audit_core(RUN_ROOT / "SPEC-CORE")
display(pd.Series(core_audit, name="value").to_frame())


## 3. Runtime truth isolation

This is deployment evidence, not the primary leakage proof. The primary
proof remains each 01A original-versus-redacted translator test.

Here the same pack is materialised with and without evaluation mounted.
`SPEC-CORE` hashes and a deterministic runtime probe must match. The negative
control deliberately reads `SPEC-EVAL` and must fail when it is absent.


In [ ]:
def deliberately_leaky_scorer(run_root):
    path = Path(run_root) / "SPEC-EVAL" / "manifest.json"
    if not path.exists():
        raise FileNotFoundError("SPEC-EVAL is not mounted")
    return read_json(path)["row_counts"]


def run_runtime_isolation_test():
    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        mounted = temporary / "mounted"
        unmounted = temporary / "unmounted"
        materialise_canonical(PACK_ROOT, mounted, include_evaluation=True, as_of_ts=AS_OF_TS)
        materialise_canonical(PACK_ROOT, unmounted, include_evaluation=False, as_of_ts=AS_OF_TS)

        assert core_content_hashes(mounted / "SPEC-CORE") == core_content_hashes(unmounted / "SPEC-CORE")
        assert runtime_probe(mounted / "SPEC-CORE") == runtime_probe(unmounted / "SPEC-CORE")
        deliberately_leaky_scorer(mounted)
        try:
            deliberately_leaky_scorer(unmounted)
        except FileNotFoundError:
            pass
        else:
            raise AssertionError("Negative control unexpectedly read unmounted truth")

if pack_manifest["evaluation_tables"]:
    run_runtime_isolation_test()
    isolation_status = "pass"
    print("PASS — SPEC-CORE and runtime output are truth-invariant")
    print("PASS — the negative control fails without SPEC-EVAL")
else:
    isolation_status = "not_run_no_evaluation"
    print("NOT RUN — runtime truth isolation requires a labelled development pack")


## 4. Temporal isolation

A small generic pack fixture is created twice: once with future observations
present and once physically truncated. Both are materialised at the same
cutoff. Their canonical content must match, including observation-derived
entity bounds. This catches accidental use of future rows.


In [ ]:
def write_pack_fixture(source_pack, destination, cutoff=None):
    source_pack, destination = Path(source_pack), Path(destination)
    source_manifest = read_json(source_pack / "source_manifest.json")
    pack_manifest = read_json(source_pack / "pack_manifest.json")
    with tempfile.TemporaryDirectory() as staging_name:
        staging = Path(staging_name) / "pack"
        core = staging / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)

        catalogue = pd.read_parquet(source_pack / "PACK-CORE" / "metric_catalogue.parquet")
        first_part = sorted((source_pack / "PACK-CORE" / "observations").glob("part-*.parquet"))[0]
        sample = pd.read_parquet(first_part).sort_values("event_ts").head(2_000)
        sample["event_ts"] = pd.to_datetime(sample["event_ts"], utc=True)
        if cutoff is not None:
            sample = sample.loc[sample["event_ts"].le(cutoff)]
        if sample.empty:
            raise ValueError("Temporal fixture is empty")
        sample.to_parquet(observations / "part-00000.parquet", index=False)

        registry = pd.read_parquet(source_pack / "PACK-CORE" / "entity_registry.parquet")
        registry = registry.loc[registry["entity_id"].astype(str).isin(sample["entity_id"].astype(str).unique())]
        episodes = pd.read_parquet(source_pack / "PACK-CORE" / "observation_episodes.parquet")
        episodes = episodes.loc[
            episodes["episode_id"].astype(str).isin(sample["episode_id"].astype(str).unique())
        ]
        catalogue.to_parquet(core / "metric_catalogue.parquet", index=False)
        registry[PACK_ENTITY_SCHEMA].to_parquet(core / "entity_registry.parquet", index=False)
        episodes[PACK_EPISODE_SCHEMA].to_parquet(
            core / "observation_episodes.parquet", index=False
        )

        finalise_pack(
            staging,
            sector=pack_manifest["sector"],
            pack_version="temporal-fixture-v1",
            source_manifest={"fixture_of": source_manifest["source_id"]},
        )
        shutil.copytree(staging, destination)
    return sample


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    preview = pd.read_parquet(
        sorted((PACK_ROOT / "PACK-CORE" / "observations").glob("part-*.parquet"))[0],
        columns=["event_ts"],
    ).sort_values("event_ts").head(2_000)
    cutoff = pd.to_datetime(preview["event_ts"], utc=True).iloc[len(preview) // 2]

    full_fixture = temporary / "full_pack"
    truncated_fixture = temporary / "truncated_pack"
    write_pack_fixture(PACK_ROOT, full_fixture)
    write_pack_fixture(PACK_ROOT, truncated_fixture, cutoff=cutoff)

    full_run = temporary / "full_as_of"
    truncated_run = temporary / "truncated_as_of"
    materialise_canonical(full_fixture, full_run, include_evaluation=False, as_of_ts=cutoff)
    materialise_canonical(truncated_fixture, truncated_run, include_evaluation=False, as_of_ts=cutoff)
    assert core_content_hashes(full_run / "SPEC-CORE") == core_content_hashes(truncated_run / "SPEC-CORE")

print("PASS — observations appended after as_of_ts cannot change earlier canonical content")


## 5. Acceptance report and output inspection

The final block prints all small tables and manifests. Partitioned telemetry
is summarized rather than printed in full.


In [ ]:
acceptance = {
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "pack_valid": True,
    "truth_isolation": isolation_status,
    "negative_control": isolation_status,
    "temporal_isolation": "pass",
    "adapter_sector_branch": False,
    "tier0_tables": list(CORE_SCHEMAS),
}
write_json(RUN_ROOT / "acceptance_report.json", acceptance)
display(pd.Series(acceptance, name="result").to_frame())

for name in (
    "metric_catalogue", "entity_registry",
    "observation_episodes", "collection_gaps",
):
    path = RUN_ROOT / "SPEC-CORE" / f"{name}.parquet"
    frame = pd.read_parquet(path)
    print(f"\n{name}: {len(frame):,} rows")
    display(frame.head(10))

telemetry_parts = sorted((RUN_ROOT / "SPEC-CORE" / "telemetry").glob("part-*.parquet"))
print(f"\ntelemetry: {len(telemetry_parts):,} parts")
display(pd.read_parquet(telemetry_parts[0]).head(10))
display(pd.Series(read_json(RUN_ROOT / "workflow_report.json"), name="value").to_frame())
print("Canonical run:", RUN_ROOT)
print("Next: 02_CANONICAL_EDA.ipynb")
